In [1]:
import os
os.chdir('/home/smallyan/eval_agent')
print("Working directory set to:", os.getcwd())

Working directory set to: /home/smallyan/eval_agent


# Generalizability Evaluation for Function Vectors Repository

## Overview
This notebook evaluates the generalizability of the findings in the `/net/scratch2/smallyan/function_vectors_eval` repository.

We will evaluate three criteria:
1. **GT1: Model Generalization** - Does the finding transfer to a new model?
2. **GT2: Data Generalization** - Does the finding hold on new data instances?
3. **GT3: Method Generalization** - Can the method be applied to another similar task?

In [2]:
# First, let's explore the repository structure
import subprocess

repo_path = "/net/scratch2/smallyan/function_vectors_eval"
result = subprocess.run(['find', repo_path, '-type', 'f', '-name', '*.py'], capture_output=True, text=True)
print("Python files:")
print(result.stdout)

result2 = subprocess.run(['find', repo_path, '-type', 'f', '-name', '*.json'], capture_output=True, text=True)
print("\nJSON files:")
print(result2.stdout)

result3 = subprocess.run(['find', repo_path, '-type', 'f', '-name', '*.md'], capture_output=True, text=True)
print("\nMarkdown files:")
print(result3.stdout)

Python files:
/net/scratch2/smallyan/function_vectors_eval/src/portability_eval.py
/net/scratch2/smallyan/function_vectors_eval/src/utils/eval_utils.py
/net/scratch2/smallyan/function_vectors_eval/src/utils/prompt_utils.py
/net/scratch2/smallyan/function_vectors_eval/src/utils/intervention_utils.py
/net/scratch2/smallyan/function_vectors_eval/src/utils/extract_utils.py
/net/scratch2/smallyan/function_vectors_eval/src/utils/__init__.py
/net/scratch2/smallyan/function_vectors_eval/src/utils/model_utils.py
/net/scratch2/smallyan/function_vectors_eval/src/test_numheads.py
/net/scratch2/smallyan/function_vectors_eval/src/compute_indirect_effect.py
/net/scratch2/smallyan/function_vectors_eval/src/vocab_reconstruction.py
/net/scratch2/smallyan/function_vectors_eval/src/__init__.py
/net/scratch2/smallyan/function_vectors_eval/src/eval_scripts/fv_eval_sweep.py
/net/scratch2/smallyan/function_vectors_eval/src/compute_avg_hidden_state.py
/net/scratch2/smallyan/function_vectors_eval/src/natural_te

In [3]:
# Let's read the plan.md to understand the research findings
with open("/net/scratch2/smallyan/function_vectors_eval/plan.md", "r") as f:
    plan_content = f.read()
print(plan_content[:5000])

# Plan
## Objective
To investigate whether autoregressive transformer language models develop compact vector representations of input-output functions (called function vectors) within their hidden states during in-context learning, and to characterize these representations across diverse tasks and models.

## Hypothesis
1. A small number of attention heads transport a compact representation of the demonstrated task (function vector) that is robust to changes in context and can trigger task execution in zero-shot and natural text settings.
2. Function vectors contain information encoding the output space of the function, but this information alone is not sufficient to reconstruct a working function vector.
3. Function vectors can be composed through vector algebra to create vectors that trigger new complex tasks combining constituent tasks.

## Methodology
1. Apply causal mediation analysis to identify attention heads with highest average indirect effect (AIE) across diverse ICL tasks. 

In [4]:
# Let's look at the directory structure
result = subprocess.run(['ls', '-la', repo_path], capture_output=True, text=True)
print(result.stdout)

# Check the evaluation folder
result2 = subprocess.run(['ls', '-la', f"{repo_path}/evaluation"], capture_output=True, text=True)
print("\nEvaluation folder:")
print(result2.stdout)

total 2349
drwxrwx---  7 smallyan smallyan      13 Jan  7 17:11 .
drwxrwx--- 37 smallyan smallyan      37 Jan  7 19:41 ..
-rw-rw----  1 smallyan smallyan    2101 Dec 21 20:30 CodeWalkthrough.md
drwxrwx---  5 smallyan smallyan       6 Dec 23 02:11 dataset_files
-rw-r-----  1 smallyan smallyan 3236102 Dec 21 20:33 documentation.pdf
drwxrwx---  2 smallyan smallyan       4 Jan  8 09:06 evaluation
-rw-rw----  1 smallyan smallyan     543 Dec 21 20:30 fv_environment.yml
-rw-rw----  1 smallyan smallyan   26597 Dec 21 20:30 fv_overview.png
drwxrwx---  7 smallyan smallyan      14 Dec 23 02:28 .git
-rw-rw----  1 smallyan smallyan     294 Dec 21 20:30 .gitignore
drwxrwx---  2 smallyan smallyan       3 Dec 21 20:30 notebooks
-rw-rw----  1 smallyan smallyan    4291 Dec 21 20:32 plan.md
drwxrwx---  5 smallyan smallyan      14 Dec 21 21:14 src


Evaluation folder:
total 36
drwxrwx--- 2 smallyan smallyan     4 Jan  8 09:06 .
drwxrwx--- 7 smallyan smallyan    13 Jan  7 17:11 ..
-rw-rw---- 1 smallyan sma

In [5]:
# Read the existing consistency evaluation to understand what was done
with open(f"{repo_path}/evaluation/consistency_evaluation.json", "r") as f:
    consistency_eval = f.read()
print(consistency_eval)

{
    "Checklist": {
        "CS1_Results_vs_Conclusion": "PASS",
        "CS2_Plan_vs_Implementation": "FAIL"
    },
    "Rationale": {
        "CS1_Results_vs_Conclusion": "All evaluable conclusions in the documentation are consistent with the code implementation. The methodology is correctly implemented in compute_indirect_effect.py, extract_utils.py, evaluate_function_vector.py, portability_eval.py, natural_text_eval.py, and vocab_reconstruction.py. The pre-computed top_heads in extract_utils.py match the documentation's description of using 10-100 heads based on model size. No contradictions were found between the documentation and the implementation.",
        "CS2_Plan_vs_Implementation": "Plan Step 4 (Vector Algebra Composition) is missing from the implementation. While Steps 1-3 are fully implemented, there is no dedicated script for inter-task FV composition (Eq. 7-8 from the paper). The experiments described in Table 7 (Last-Antonym, Last-Capitalize, Last-Country-Capital, et

In [6]:
# Let's explore the key implementation files to understand how function vectors work
with open(f"{repo_path}/src/utils/extract_utils.py", "r") as f:
    extract_utils_content = f.read()
print(extract_utils_content[:4000])

import os, re, json

import torch, numpy as np
import pandas as pd
from baukit import TraceDict

# Include prompt creation helper functions
from .prompt_utils import *
from .intervention_utils import *
from .model_utils import *
from .eval_utils import *


# Attention Activations
def gather_attn_activations(prompt_data, layers, dummy_labels, model, tokenizer, model_config):
    """
    Collects activations for an ICL prompt 

    Parameters:
    prompt_data: dict containing ICL prompt examples, and template information
    layers: layer names to get activatons from
    dummy_labels: labels and indices for a baseline prompt with the same number of example pairs
    model: huggingface model
    tokenizer: huggingface tokenizer

    Returns:
    td: tracedict with stored activations
    idx_map: map of token indices to respective averaged token indices
    idx_avg: dict containing token indices of multi-token words
    """   
    
    # Get sentence and token labels
    query = prompt_dat

In [7]:
# Continue reading extract_utils.py
print(extract_utils_content[4000:8000])

and separators is not None:
            prompt_data = word_pairs_to_prompt_data(word_pairs, query_target_pair=word_pairs_test, prepend_bos_token=prepend_bos, 
                                                    shuffle_labels=shuffle_labels, prefixes=prefixes, separators=separators)
        else:
            prompt_data = word_pairs_to_prompt_data(word_pairs, query_target_pair=word_pairs_test, prepend_bos_token=prepend_bos, shuffle_labels=shuffle_labels)
        activations_td,idx_map,idx_avg = gather_attn_activations(prompt_data=prompt_data, 
                                                            layers = model_config['attn_hook_names'], 
                                                            dummy_labels=dummy_labels, 
                                                            model=model, 
                                                            tokenizer=tokenizer, 
                                                            model_config=model_config)
        
       

In [8]:
# Continue reading to find the top_heads configuration
print(extract_utils_content[8000:12000])

set,n_test_examples, replace=False)]
        if prefixes is not None and separators is not None:
            prompt_data = word_pairs_to_prompt_data(word_pairs, query_target_pair=word_pairs_test, prepend_bos_token=prepend_bos, 
                                                    shuffle_labels=shuffle_labels, prefixes=prefixes, separators=separators)
        else:
            prompt_data = word_pairs_to_prompt_data(word_pairs, query_target_pair=word_pairs_test, prepend_bos_token=prepend_bos, shuffle_labels=shuffle_labels)
        activations_td = gather_layer_activations(prompt_data=prompt_data, 
                                                  layers = model_config['layer_hook_names'], 
                                                  model=model, 
                                                  tokenizer=tokenizer, 
                                                  model_config=model_config)
        
        stack_initial = torch.vstack([activations_td[layer].output[0] for layer 

In [9]:
# Let's look for top_heads configuration in the code
result = subprocess.run(['grep', '-r', 'top_heads', repo_path], capture_output=True, text=True)
print(result.stdout)

/net/scratch2/smallyan/function_vectors_eval/src/portability_eval.py:    parser.add_argument('--n_top_heads', help='Number of attenion head outputs used to compute function vector', required=False, type=int, default=10)
/net/scratch2/smallyan/function_vectors_eval/src/portability_eval.py:    n_top_heads = args.n_top_heads
/net/scratch2/smallyan/function_vectors_eval/src/portability_eval.py:    fv, top_heads = compute_universal_function_vector(mean_activations, model, model_config=model_config, n_top_heads=n_top_heads)   
/net/scratch2/smallyan/function_vectors_eval/src/utils/extract_utils.py:def compute_function_vector(mean_activations, indirect_effect, model, model_config, n_top_heads = 10, token_class_idx=-1):
/net/scratch2/smallyan/function_vectors_eval/src/utils/extract_utils.py:        n_top_heads: The number of heads to use when computing the summed function vector
/net/scratch2/smallyan/function_vectors_eval/src/utils/extract_utils.py:        top_heads: list of the top influenti

In [10]:
# Let's read the compute_universal_function_vector function to understand the supported models
# Search for the function definition
result = subprocess.run(['grep', '-A', '50', 'def compute_universal_function_vector', f"{repo_path}/src/utils/extract_utils.py"], 
                       capture_output=True, text=True)
print(result.stdout)

def compute_universal_function_vector(mean_activations, model, model_config, n_top_heads=10):
    """
        Computes a "function vector" vector that communicates the task observed in ICL examples used for downstream intervention
        using the set of heads with universally highest causal effect computed across a set of ICL tasks
        
        Parameters:
        mean_activations: tensor of size (Layers, Heads, Tokens, head_dim) containing the average activation of each head for a particular task
        model: huggingface model being used
        model_config: contains model config information (n layers, n heads, etc.)
        n_top_heads: The number of heads to use when computing the function vector

        Returns:
        function_vector: vector representing the communication of a particular task
        top_heads: list of the top influential heads represented as tuples [(L,H,S), ...], (L=Layer, H=Head, S=Avg. Indirect Effect Score)         
    """
    model_resid_dim = mo

In [11]:
# Let's also check model_utils.py to see what models are supported
with open(f"{repo_path}/src/utils/model_utils.py", "r") as f:
    model_utils_content = f.read()
print(model_utils_content)

import torch
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer, LlamaTokenizer, LlamaForCausalLM
import os
import random
from typing import *


def load_gpt_model_and_tokenizer(model_name:str, device='cuda', revision=None):
    """
    Loads a huggingface model and its tokenizer

    Parameters:
    model_name: huggingface name of the model to load (e.g. GPTJ: "EleutherAI/gpt-j-6B", or "EleutherAI/gpt-j-6b")
    device: 'cuda' or 'cpu'
    
    Returns:
    model: huggingface model
    tokenizer: huggingface tokenizer
    MODEL_CONFIG: config variables w/ standardized names
    
    """
    assert model_name is not None

    print("Loading: ", model_name)

    if model_name == 'gpt2-xl':
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        tokenizer.pad_token = tokenizer.eos_token
        model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

        MODEL_CONFIG={"n_heads":model.config.n_head,
                      "n_layers

In [12]:
# Look at a sample dataset to understand the data format
import json
with open(f"{repo_path}/dataset_files/abstractive/antonym.json", "r") as f:
    antonym_data = json.load(f)
print("Antonym dataset structure:")
print("Keys:", antonym_data.keys())
print("\nTrain samples (first 5):")
for i, sample in enumerate(antonym_data['train'][:5]):
    print(f"  {i}: {sample}")
print("\nValid samples (first 5):")
for i, sample in enumerate(antonym_data['valid'][:5]):
    print(f"  {i}: {sample}")

Antonym dataset structure:


AttributeError: 'list' object has no attribute 'keys'

In [13]:
# Let's see the actual structure of the data
print("Antonym dataset structure:")
print("Type:", type(antonym_data))
print("Length:", len(antonym_data))
print("\nFirst 10 samples:")
for i, sample in enumerate(antonym_data[:10]):
    print(f"  {i}: {sample}")

Antonym dataset structure:
Type: <class 'list'>
Length: 2398

First 10 samples:
  0: {'input': 'flawed', 'output': 'perfect'}
  1: {'input': 'orthodox', 'output': 'unorthodox'}
  2: {'input': 'true', 'output': 'false'}
  3: {'input': 'daily', 'output': 'nightly'}
  4: {'input': 'distribution', 'output': 'concentration'}
  5: {'input': 'valid', 'output': 'invalid'}
  6: {'input': 'expand', 'output': 'contract'}
  7: {'input': 'practical', 'output': 'impractical'}
  8: {'input': 'privilege', 'output': 'disadvantage'}
  9: {'input': 'mammoth', 'output': 'tiny'}


In [14]:
# Let's look at evaluate_function_vector.py to understand how FV evaluation works
with open(f"{repo_path}/src/evaluate_function_vector.py", "r") as f:
    eval_fv_content = f.read()
print(eval_fv_content[:4000])

import os, json
import torch, numpy as np
import argparse

# Include prompt creation helper functions
from utils.prompt_utils import *
from utils.intervention_utils import *
from utils.model_utils import *
from utils.eval_utils import *
from utils.extract_utils import *
from compute_indirect_effect import compute_indirect_effect

if __name__ == "__main__":
    
    parser = argparse.ArgumentParser()

    parser.add_argument('--dataset_name', help='Name of the dataset to be loaded', type=str, required=True)
    parser.add_argument('--n_top_heads', help='Number of attenion head outputs used to compute function vector', required=False, type=int, default=10)
    parser.add_argument('--edit_layer', help='Layer for intervention. If -1, sweep over all layers', type=int, required=False, default=-1) # 
    parser.add_argument('--model_name', help='Name of model to be loaded', type=str, required=False, default='EleutherAI/gpt-j-6b')
    parser.add_argument('--root_data_dir', help='Root directory

## Understanding the Research

Based on the exploration:

1. **Main Finding**: The research proposes "Function Vectors" - compact vector representations that capture input-output functions during in-context learning in transformer language models.

2. **Models Used in Original Work**: 
   - GPT-J 6B (EleutherAI/gpt-j-6b)
   - GPT-NeoX 20B
   - Llama-2-7B, Llama-2-13B, Llama-2-70B

3. **Key Method**: Causal mediation analysis identifies top attention heads with highest Average Indirect Effect (AIE). Function vectors are computed by summing the task-conditioned mean outputs of these top causal attention heads.

4. **Tasks**: Over 40 diverse ICL tasks including antonym, synonym, translation, country-capital, sentiment, etc.

---

## Generalizability Evaluation Plan

### GT1: Model Generalization
Test if the function vector finding works on a model NOT used in the original work. The code supports:
- GPT-2-XL
- Pythia models
- Gemma models
- OLMo models

We will try Pythia-1.4B or Gemma-2B as a new model.

### GT2: Data Generalization  
Test if the function vector works on NEW data instances not in the original dataset.

### GT3: Method Generalization
Test if the causal mediation + function vector extraction method can be applied to a different similar task.

In [15]:
# Check CUDA availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

CUDA available: True
CUDA device: NVIDIA A100 80GB PCIe
CUDA memory: 79.25 GB


In [16]:
# Set up the path and import utilities
import sys
sys.path.insert(0, f"{repo_path}/src")

from utils.model_utils import load_gpt_model_and_tokenizer, set_seed
from utils.extract_utils import get_mean_head_activations, compute_universal_function_vector, compute_function_vector
from utils.prompt_utils import word_pairs_to_prompt_data, load_dataset
from utils.intervention_utils import function_vector_intervention
from utils.eval_utils import n_shot_eval_no_intervention

set_seed(42)
print("Utilities imported successfully")

Utilities imported successfully


---

# GT1: Model Generalization Evaluation

We will test if the function vector finding generalizes to a **new model not used in the original work**.

Original models used:
- GPT-J 6B
- GPT-NeoX 20B  
- Llama-2-7B, Llama-2-13B, Llama-2-70B

**New model to test**: Pythia-1.4B (EleutherAI/pythia-1.4b) - a model from the same family (EleutherAI) but not included in the original study.

In [17]:
# Load a new model not used in original work: Pythia-1.4B
# This model is from the EleutherAI family but was NOT used in the original study
model_name = "EleutherAI/pythia-1.4b"
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Loading model: {model_name}")
model, tokenizer, model_config = load_gpt_model_and_tokenizer(model_name, device=device)
print(f"\nModel loaded successfully!")
print(f"Model config: {model_config}")

Loading model: EleutherAI/pythia-1.4b
Loading:  EleutherAI/pythia-1.4b


`torch_dtype` is deprecated! Use `dtype` instead!


OSError: PermissionError at /net/projects2/chai-lab/shared_models/hub/.locks/models--EleutherAI--pythia-1.4b/ad4388632e922d0c58c23cb315292d516deb70af0573f9c1543ce84158bff59c.lock when downloading EleutherAI/pythia-1.4b. Check cache directory permissions. Common causes: 1) another user is downloading the same model (please wait); 2) a previous download was canceled and the lock file needs manual removal.

In [18]:
# Set a different cache directory to avoid permission issues
import os
os.environ['HF_HOME'] = '/home/smallyan/.cache/huggingface'
os.environ['TRANSFORMERS_CACHE'] = '/home/smallyan/.cache/huggingface/transformers'

# Try with GPT-2-XL which is smaller and likely cached
model_name = "gpt2-xl"
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Loading model: {model_name}")
model, tokenizer, model_config = load_gpt_model_and_tokenizer(model_name, device=device)
print(f"\nModel loaded successfully!")
print(f"Model config: {model_config}")

Loading model: gpt2-xl
Loading:  gpt2-xl


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/6.43G [00:00<?, ?B/s]

In [19]:
# Check if model is loaded
print(f"Model type: {type(model)}")
print(f"Model config: {model_config}")

In [20]:
# Model seems to still be loading, let's check again
import time
time.sleep(5)
try:
    print(f"Model loaded: {model is not None}")
    print(f"Model config: {model_config}")
except:
    print("Model still loading...")

In [21]:
# Check model state
print("Checking model...")
print(f"model_config = {model_config}")

In [22]:
print("Testing output")